# model_inference.ipynb

Loads the **best registered model** (XGBoost in my runs) directly from the
MLflow Model Registry and runs `predict_proba` on the **raw** test set.
No manual preprocessing — the saved sklearn `Pipeline` does it all.

## 0. Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")
from src.preprocessing import load_raw

import mlflow
import mlflow.sklearn

try:
    import dagshub
    dagshub.init(
        repo_owner=os.environ.get("DAGSHUB_USER", "gurtm23"),
        repo_name=os.environ.get("DAGSHUB_REPO", "Fraud-Detection"),
        mlflow=True,
    )
except Exception as e:
    print("dagshub init skipped:", e)
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns"))


## 1. Load raw test set

Notice we do NOT preprocess anything here — the pipeline pulled from the
registry already contains the cleaning + FE + FS steps.

In [ ]:
DATA_PATH = os.environ.get("IEEE_DATA", "../data")
_, test = load_raw(DATA_PATH)
print("test:", test.shape)

X_test = test.drop(columns=["TransactionID"])


## 2. Pull best model from the Registry

I registered the XGBoost pipeline as `ieee_fraud_best`. DagsHub's MLflow doesn't
support stage transitions, so we use the **alias** mechanism (`production`) instead.

In [ ]:
MODEL_NAME = os.environ.get("MODEL_NAME", "ieee_fraud_best")
ALIAS = os.environ.get("MODEL_ALIAS", "production")

# alias-based URI (mlflow >= 2.6). fallback to /Production stage for older servers.
try:
    pipe = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@{ALIAS}")
    print("loaded via alias:", ALIAS)
except Exception:
    pipe = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")
    print("loaded via stage Production")
print(pipe)


## 3. Predict on raw test data

In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
print("predictions:", proba.shape, "  range:", proba.min(), proba.max())


## 4. Build the Kaggle submission

In [ ]:
sub = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": proba,
})
sub.to_csv("submission.csv", index=False)
sub.head()


In [ ]:
# kaggle command (run in terminal, not here):
# kaggle competitions submit -c ieee-fraud-detection -f submission.csv -m "best XGBoost pipeline"


## 5. (Optional) — register a new model from a run

After all 5 model notebooks finish, find the best `*_FinalFit` run by `cv_mean_auc`
in the MLflow UI, copy its **model_uri** (printed in the notebook output —
looks like `models:/m-xxxxxx`) and run this cell to register & promote.

In [ ]:
# from mlflow.tracking import MlflowClient
#
# client = MlflowClient()
# BEST_MODEL_URI = "models:/m-PASTE-FROM-FINALFIT-OUTPUT"
#
# result = mlflow.register_model(model_uri=BEST_MODEL_URI, name=MODEL_NAME)
# print("registered version:", result.version)
#
# # promote to production via alias (works on DagsHub MLflow)
# client.set_registered_model_alias(MODEL_NAME, "production", result.version)
# print("alias 'production' now points to version", result.version)
